In [ ]:
import os
import logging
from dataclasses import dataclass
from typing import Dict, List, TypedDict

from dotenv import load_dotenv
from IPython.display import Image
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.pydantic_v1 import BaseModel
from langchain_core.runnables import RunnableConfig
from langchain_ibm import WatsonxLLM
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.store.memory import InMemoryStore
from tavily import TavilyClient
import uuid

from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import DecodingMethods

# --- Configuration and Environment Setup ---

load_dotenv()

# Logging setup
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Watsonx.ai configuration
watsonx_api_key = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("PROJECT_ID")
url = os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com") # Default URL

# Tavily configuration
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not watsonx_api_key:
    raise ValueError("Please set the WATSONX_API_KEY in your .env file.")
if not project_id:
    raise ValueError("Please set the PROJECT_ID in your .env file.")
if not tavily_api_key:
    raise ValueError("Please set the TAVILY_API_KEY in your .env file.")

parameters = {
    GenParams.DECODING_METHOD: DecodingMethods.SAMPLE,
    GenParams.MAX_NEW_TOKENS: 500,
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.TEMPERATURE: 0.7,
    GenParams.TOP_K: 50,
    GenParams.TOP_P: 1,
}

# --- Watsonx.ai LLM Initialization ---

model = WatsonxLLM(
    model_id="meta-llama/llama-3-70b-instruct",
    url=url,
    apikey=watsonx_api_key,
    project_id=project_id,
    params=parameters,
)

# --- Agent State Definition ---

class AgentState(TypedDict):
    """
    A dictionary representing the state of the research agent.

    Attributes:
        task (str): The description of the task to be performed.
        plan (str): The research plan generated for the task.
        draft (str): The current draft of the research report.
        critique (str): The critique received for the draft.
        content (List[str]): A list of content gathered during research.
        editor_comment (str): Comments from the editor
        revision_number (int): The current revision number of the draft.
        max_revisions (int): The maximum number of revisions allowed.
        finalized_state (bool): Indicates whether the report is finalized.
    """

    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    editor_comment: str
    revision_number: int
    max_revisions: int
    finalized_state: bool

# --- Research Prompts ---

@dataclass
class ResearchPlanPrompt:
    system_template: str = """
    You are an expert writer tasked with creating a high-level outline for a research report.
    Write such an outline for the user-provided topic. Include relevant notes or instructions for each section.
    The style of the research report should be geared towards the educated public. It should be detailed enough to provide
    a good level of understanding of the topic, but not unnecessarily dense. Think of it more like a whitepaper to be consumed
    by a business leader rather than an academic journal article.
    """

@dataclass
class GenerationPrompt:
    system_template: str = """
    You are a researcher tasked with creating a research report draft based on a user-provided plan.
    Use the provided content to create a draft of the research report.
    The style of the research report should be geared towards the educated public. It should be detailed enough to provide
    a good level of understanding of the topic, but not unnecessarily dense. Think of it more like a whitepaper to be consumed
    by a business leader rather than an academic journal article.
    """

@dataclass
class ReviewPrompt:
    system_template: str = """
    You are a reviewer tasked with reviewing a research report draft based on a user-provided plan.
    Review the provided draft and provide feedback on how to improve it.
    """

@dataclass
class CritiquePrompt:
    system_template: str = """
    You are a critic tasked with critiquing a research report draft based on a user-provided plan.
    Review the provided draft and provide a critique of the draft.
    """

@dataclass
class EditorPrompt:
    system_template: str = """
    You are an editor tasked with reviewing a research report draft based on a user-provided plan.
    Review the provided draft and provide a critique of the draft.
    Based on the critique, decide whether the draft should be accepted, rejected, or sent for revision.
    If the draft is accepted, set the finalized_state to True.
    If the draft is rejected, set the finalized_state to True.
    If the draft is sent for revision, increment the revision_number by 1.
    """

@dataclass
class RejectPrompt:
    system_template: str = """
    The research report draft has been rejected.
    """

@dataclass
class AcceptPrompt:
    system_template: str = """
    The research report draft has been accepted.
    """

# --- Agent Nodes ---

class AgentNodes:
    def __init__(self, model: WatsonxLLM, searcher: TavilyClient):
        self.model = model
        self.searcher = searcher

    def plan_node(self, state: AgentState) -> Dict[str, str]:
        """
        Generate a research plan based on the current state.
        """
        try:
            messages = [
                SystemMessage(content=ResearchPlanPrompt.system_template),
                HumanMessage(content=state["task"]),
            ]
            response = self.model.invoke(messages)
            return {"plan": response}
        except Exception as e:
            logger.error(f"Error in plan_node: {e}")
            return {"plan": "Error generating research plan."}

    def research_plan_node(self, state: AgentState) -> Dict[str, List[str]]:
        """
        Perform research based on the generated plan.
        """
        try:
            plan = state["plan"]
            queries = plan.split("\n")
            content = []
            for query in queries:
                search_result = self.searcher.search(query=query, search_depth="advanced")
                if search_result and search_result.results:
                    for result in search_result.results:
                        content.append(result.content)
                else:
                    logger.warning(f"No search results found for query: {query}")

            return {"content": content}
        except Exception as e:
            logger.error(f"Error in research_plan_node: {e}")
            return {"content": []}

    def generation_node(self, state: AgentState) -> Dict[str, str]:
        """
        Generate a draft based on the research content and plan.
        """
        try:
            messages = [
                SystemMessage(content=GenerationPrompt.system_template),
                HumanMessage(
                    content=f"Plan: {state['plan']}\n\nContent: {state['content']}"
                ),
            ]
            response = self.model.invoke(messages)
            return {"draft": response}
        except Exception as e:
            logger.error(f"Error in generation_node: {e}")
            return {"draft": "Error generating draft."}

    def review_node(self, state: AgentState) -> Dict[str, str]:
        """
        Review the generated draft.
        """
        try:
            messages = [
                SystemMessage(content=ReviewPrompt.system_template),
                HumanMessage(
                    content=f"Plan: {state['plan']}\n\nDraft: {state['draft']}"
                ),
            ]
            response = self.model.invoke(messages)
            return {"review": response}
        except Exception as e:
            logger.error(f"Error in review_node: {e}")
            return {"review": "Error reviewing draft."}

    def research_critique_node(self, state: AgentState) -> Dict[str, str]:
        """
        Critique the generated draft and research content.
        """
        try:
            messages = [
                SystemMessage(content=CritiquePrompt.system_template),
                HumanMessage(
                    content=f"Plan: {state['plan']}\n\nDraft: {state['draft']}\n\nContent: {state['content']}"
                ),
            ]
            response = self.model.invoke(messages)
            return {"critique": response}
        except Exception as e:
            logger.error(f"Error in research_critique_node: {e}")
            return {"critique": "Error critiquing draft."}

    def editor_node(self, state: AgentState) -> Dict[str, str]:
        """
        Simulate an editor reviewing and making decisions about the draft.
        """
        try:
            messages = [
                SystemMessage(content=EditorPrompt.system_template),
                HumanMessage(
                    content=f"Plan: {state['plan']}\n\nDraft: {state['draft']}\n\nCritique: {state['critique']}"
                ),
            ]
            response = self.model.invoke(messages)

            # Determine the next state based on the editor's response
            if "accept" in response.lower():
                finalized_state = True
                next_state = "accepted"
            elif "reject" in response.lower():
                finalized_state = True
                next_state = "rejected"
            else:
                finalized_state = False
                next_state = "to_review"
                state["revision_number"] += 1

            state["finalized_state"] = finalized_state
            state["editor_comment"] = response

            return {
                "editor_comment": response,
                "finalized_state": finalized_state,
                "next_state": next_state
            }

        except Exception as e:
            logger.error(f"Error in editor_node: {e}")
            return {
                "editor_comment": "Error in editor review.",
                "finalized_state": False,  # Default to False on error
                "next_state": "to_review"  # Default to sending for review on error
            }

    def reject_node(self, state: AgentState) -> Dict[str, str]:
        """
        Handle the rejection of the draft.
        """
        try:
            messages = [
                SystemMessage(content=RejectPrompt.system_template),
            ]
            response = self.model.invoke(messages)
            return {"reject_message": response}
        except Exception as e:
            logger.error(f"Error in reject_node: {e}")
            return {"reject_message": "Draft rejected."}

    def accept_node(self, state: AgentState) -> Dict[str, str]:
        """
        Handle the acceptance of the draft.
        """
        try:
            messages = [
                SystemMessage(content=AcceptPrompt.system_template),
            ]
            response = self.model.invoke(messages)
            return {"accept_message": response}
        except Exception as e:
            logger.error(f"Error in accept_node: {e}")
            return {"accept_message": "Draft accepted."}

# --- Agent Edges ---

class AgentEdges:
    def should_continue(self, state: AgentState) -> str:
        """
        Determine whether the research process should continue based on the current state.
        """
        next_state = state.get("next_state", "to_review")

        if next_state == "accepted":
            return "accepted"
        elif next_state == "rejected":
            return "rejected"
        elif state["revision_number"] > state["max_revisions"]:
            logger.info("Revision number > max allowed revisions")
            return "rejected"
        else:
            return "to_review"

# --- State Graph Workflow Definition ---

searcher = TavilyClient(api_key=tavily_api_key)
nodes = AgentNodes(model, searcher)
edges = AgentEdges()

agent = StateGraph(AgentState)

# Add nodes
agent.add_node("initial_plan", nodes.plan_node)
agent.add_node("do_research", nodes.research_plan_node)
agent.add_node("write", nodes.generation_node)
agent.add_node("review", nodes.review_node)
agent.add_node("research_revise", nodes.research_critique_node)
agent.add_node("editor", nodes.editor_node)
agent.add_node("reject", nodes.reject_node)
agent.add_node("accept", nodes.accept_node)


# Add edges
agent.set_entry_point("initial_plan")
agent.add_edge("initial_plan", "do_research")
agent.add_edge("do_research", "write")
agent.add_edge("write", "editor")
agent.add_conditional_edges(
    "editor",
    edges.should_continue,
    {
        "accepted": "accept",
        "rejected": "reject",
        "to_review": "review"
    },
)
agent.add_edge("review", "research_revise")
agent.add_edge("research_revise", "write")
agent.add_edge("reject", END)
agent.add_edge("accept", END)

# --- Graph Compilation and Execution ---

# Compile the graph
checkpointer = MemorySaver()
in_memory_store = InMemoryStore()
graph = agent.compile(checkpointer=checkpointer, store=in_memory_store)

# Visualize the graph
try:
    Image(graph.get_graph().draw_png())
except:
    print("Unable to display graph architecture. Ensure that you have installed the required dependencies.")

def execute_research_task(
    task_description: str, max_revisions: int = 3
) -> List[Dict]:
    """
    Executes the research task workflow.

    Args:
        task_description (str): The description of the research task.
        max_revisions (int): The maximum number of revisions allowed.

    Returns:
        List[Dict]: A list of dictionaries representing the updates from each step of the workflow.
    """

    results = []

    user_id = "1" # You can make this dynamic if needed
    config = {"configurable": {"thread_id": "1", "user_id": user_id}}
    namespace = (user_id, "memories")

    try:
        for i, update in enumerate(
            graph.stream(
                {
                    "task": task_description,
                    "max_revisions": max_revisions,
                    "revision_number": 0,
                    "finalized_state": False
                },
                config,
                stream_mode="updates",
            )
        ):
            print(f"--- Update {i+1} ---")
            print(update)
            memory_id = str(uuid.uuid4())
            in_memory_store.put(namespace, memory_id, {"memory": update})
            results.append(update)
        print("--- Workflow Complete ---")
    except Exception as e:
        logger.error(f"An error occurred during workflow execution: {e}")
    return results

# --- Example Usage ---

# task_description = "What are the key trends in LLM research and application that you see in 2024"
# execute_research_task(task_description, max_revisions=1